In [ ]:
# Clone the repository
!git clone https://github.com/phanindra-max/Cross-Generator-Generalization-Project.git

# Move into the directory
%cd Cross-Generator-Generalization-Project/

# Install dependencies if you have a requirements.txt
!pip install uv
!uv pip install -r requirements.txt

# Data Pipeline: Deepfake Cross-Generator Detection

This notebook downloads and prepares all datasets for the project:
1. **FFHQ** (real faces) — 2000 thumbnails from HuggingFace
2. **FaceForensics++ FaceSwap c23** — 2000 frames from Kaggle
3. **Stable Diffusion v1.5** — 1000 generated portraits

Then creates reproducible CSV split manifests.

**Runtime**: Use a GPU runtime (T4 or better) for Stable Diffusion generation.

## 0. Install Dependencies

In [ ]:
!uv pip install -q -r requirements.txt

In [ ]:
import os
import sys
from pathlib import Path

# If running from notebooks/ directory, add project root to path
PROJECT_ROOT = Path(".").resolve().parent
if (PROJECT_ROOT / "src").exists():
    sys.path.insert(0, str(PROJECT_ROOT))
    os.chdir(PROJECT_ROOT)

print(f"Working directory: {os.getcwd()}")

## 1. Download FFHQ Real Faces (HuggingFace)

Downloads 2000 128x128 face thumbnails from the FFHQ dataset hosted on HuggingFace.

In [ ]:
from src.data.prepare_ffhq import download_ffhq

num_real = download_ffhq(
    output_dir="data/real",
    num_images=2000,
    resolution=128,
)
print(f"\nTotal real images: {num_real}")

## 2. Download FaceForensics++ FaceSwap c23 (Kaggle)

Downloads the FF++ c23 dataset from Kaggle and extracts 4 evenly-spaced frames from 500 FaceSwap videos.

**Note**: You need Kaggle credentials configured. In Colab, run:
```python
from google.colab import userdata
os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
```
Or upload your `kaggle.json` to `~/.kaggle/kaggle.json`.

In [ ]:
# Configure Kaggle credentials (uncomment the method that works for you)

# Method 1: Colab Secrets
# from google.colab import userdata
# os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
# os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")

# Method 2: Direct (replace with your credentials)
# os.environ["KAGGLE_USERNAME"] = "your_username"
# os.environ["KAGGLE_KEY"] = "your_key"

# Method 3: Upload kaggle.json
# !mkdir -p ~/.kaggle && cp kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

In [ ]:
from src.data.prepare_faceforensics import prepare_faceforensics

num_faceswap = prepare_faceforensics(
    output_dir="data/faceswap",
    num_videos=500,
    frames_per_video=4,
    resolution=128,
)
print(f"\nTotal FaceSwap frames: {num_faceswap}")

## 3. Generate Stable Diffusion v1.5 Faces

Generates 1000 portrait images using 20 diverse prompts (50 images each).
Takes ~30 minutes on a T4 GPU.

In [ ]:
from src.data.generate_diffusion import generate_diffusion_faces

num_diffusion = generate_diffusion_faces(
    output_dir="data/diffusion_sd15",
    num_images=1000,
    resolution=128,
    batch_size=4,
    seed=42,
)
print(f"\nTotal diffusion images: {num_diffusion}")

## 4. Create Split Manifests

Produces three CSV files with deterministic splits:
- `train.csv`: 1600 real + 1600 FaceSwap
- `test_indist.csv`: 400 real + 400 FaceSwap  
- `test_ood.csv`: 400 real + 1000 Stable Diffusion fakes

In [ ]:
from src.data.create_splits import create_splits

train_df, test_indist_df, test_ood_df = create_splits(
    real_dir="data/real",
    faceswap_dir="data/faceswap",
    diffusion_dir="data/diffusion_sd15",
    output_dir="data/splits",
    seed=42,
)

## 5. Verification & Sample Grid

In [ ]:
import pandas as pd

print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)

for split_name in ["train", "test_indist", "test_ood"]:
    df = pd.read_csv(f"data/splits/{split_name}.csv")
    print(f"\n{split_name}.csv: {len(df)} total samples")
    print(f"  By source: {df['source'].value_counts().to_dict()}")
    print(f"  By label:  {{0: 'real', 1: 'fake'}} -> {df['label'].value_counts().to_dict()}")

# Verify all files exist
all_dfs = pd.concat([train_df, test_indist_df, test_ood_df])
missing = [p for p in all_dfs["path"] if not Path(p).exists()]
if missing:
    print(f"\nWARNING: {len(missing)} files not found!")
    print(f"  First few: {missing[:5]}")
else:
    print(f"\nAll {len(all_dfs)} files verified to exist.")

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import random

fig, axes = plt.subplots(3, 6, figsize=(14, 7))
fig.suptitle("Sample Images by Source", fontsize=14)

sources = [("data/real", "Real (FFHQ)"), ("data/faceswap", "FaceSwap (FF++)"), ("data/diffusion_sd15", "Stable Diffusion")]

for row, (img_dir, title) in enumerate(sources):
    img_files = sorted(Path(img_dir).glob("*.png"))
    samples = random.sample(img_files, min(6, len(img_files)))
    
    for col, img_path in enumerate(samples):
        img = Image.open(img_path)
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_ylabel(title, fontsize=11, rotation=0, labelpad=80, va="center")

plt.tight_layout()
plt.savefig("results/figures/data_samples.png", dpi=150, bbox_inches="tight")
plt.show()
print("Sample grid saved to results/figures/data_samples.png")

## 6. (Optional) Copy Splits to Google Drive

If you want to persist the split CSVs across Colab sessions:

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r data/splits /content/drive/MyDrive/deepfake-cross-generator/data/
# print("Splits saved to Google Drive.")